In [1]:
%load_ext autoreload
%autoreload 2
import jax
import jax.numpy as jnp
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

from jax import jit
from jax.scipy.linalg import expm


from qdots_qll.models.single_dot_weak_coupling_GAME import *

In [2]:
m = SingleDotWeakCouplingGAME()
true_parameters


In [3]:
t = 25.4

In [4]:
key = jax.random.PRNGKey(seed=1)

In [5]:
key, subkey = jax.random.split(key)
outcome = m.generate_data(subkey, true_parameters, t, 0, 2)

In [6]:
outcome

In [8]:
lkl = m.likelihood_particle(true_parameters, t)

In [9]:
lkl[*outcome]

In [13]:
prob_state = jnp.ones(4)/4
prob_basis = jnp.ones(3)/3

In [14]:
fim = m.fim(true_parameters, t, prob_state, prob_basis)

In [23]:
lkl.shape

In [26]:
gradient = jax.jacobian(m.likelihood_particle, 0)(true_parameters, t)[:, *outcome]

In [24]:
fiminv = jnp.linalg.inv(fim)

In [27]:
fiminv@gradient

In [28]:
gradient

In [29]:
fim@gradient

In [40]:
new_par_flat = true_parameters + gradient*0.01
new_par_h = true_parameters + 0.01*fiminv@gradient

In [41]:
new_par_flat

In [42]:
new_par_h

In [43]:
true_parameters

In [44]:
m.likelihood_particle(true_parameters, t)[*outcome]

In [45]:
m.likelihood_particle(new_par_flat, t)[*outcome]

In [46]:
m.likelihood_particle(new_par_h, t)[*outcome]

In [47]:
new_par_h

In [48]:
new_par_flat

In [53]:
jax.random.normal(subkey, (4, 4))/50

In [54]:
true_parameters

In [55]:
true_parameters + jax.random.normal(subkey, (4, 4))/50

In [56]:
dist_pars = true_parameters + jax.random.normal(subkey, (4, 4))/50

In [57]:
dist_pars[0]

In [58]:
dist_pars[2]

In [59]:
jax.vmap(lambda pars: m.likelihood_particle(pars, t)[*outcome])(dist_pars)

In [60]:
m.likelihood_particle(true_parameters, t)[*outcome]

In [61]:
outcome

In [62]:
(jax.vmap(lambda pars: m.likelihood_particle(pars, t)[*outcome])(dist_pars)).sum()/4

In [63]:
p_pars = jnp.ones(4)/4

In [83]:
gradients_dist_pars = jax.vmap(lambda par: jax.jacobian(m.likelihood_particle, 0)(par, t)[:, *outcome])(dist_pars)

In [84]:
jax.vmap(lambda v: jnp.linalg.norm(v))(gradients_dist_pars)

In [85]:
grad_norms = jax.vmap(lambda v: jnp.linalg.norm(v))(gradients_dist_pars)

In [86]:
p_pars_new = p_pars + grad_norms

In [87]:
p_pars_new = p_pars_new/p_pars_new.sum()

In [88]:
p_pars_new

In [89]:
(jax.vmap(lambda pars: m.likelihood_particle(pars, t)[*outcome])(dist_pars))@ p_pars_new

In [78]:
fims_inv = jax.vmap(lambda pars: jnp.linalg.inv(m.fim(pars, t, prob_state, prob_basis)))(dist_pars)



In [91]:
jax.vmap(lambda fiminv, grad: fiminv@grad, in_axes=(0, 0))(fims_inv, gradients_dist_pars)

In [90]:
gradients_dist_pars